In [1]:
"""Copied from https://github.com/microsoft/qdk-chemistry/blob/main/examples/sossa_qre.ipynb."""

import math

import numpy as np
from qdk.qre.application import QSharpApplication
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Configuration,
    DFTHCHamiltonianContainer,
    Hamiltonian,
    MajoranaMapping,
    ModelOrbitals,
    StateVectorContainer,
    Wavefunction,
)


def get_sossa_app(num_queries_override: int = None) -> QSharpApplication:
    FE2S2_PARAMETERS = {
        "electrons": 30,
        "N": 20,
        "R": 14,
        "B": 15,
        "C": 5,
        "lambda_eff": 6.4690,
        "b_coeff": 11,
        "b_rot": 15,
    }
    TARGET_PRECISION = 1e-3
    seed = 42
    fe2s2_num_queries = math.ceil(
        math.pi * FE2S2_PARAMETERS["lambda_eff"] / (2 * TARGET_PRECISION)
    )

    if num_queries_override is not None:
        print(f"Changed num_queries from {fe2s2_num_queries} to {num_queries_override}")
        fe2s2_num_queries = num_queries_override

    n, r, b, c = (FE2S2_PARAMETERS[key] for key in ("N", "R", "B", "C"))
    rng = np.random.default_rng(seed)
    one_body = rng.standard_normal((n, n))
    basis_vectors = rng.standard_normal((r, b, n))
    basis_vectors /= np.linalg.norm(basis_vectors, axis=-1, keepdims=True)
    synthetic_fe2_s2_hamiltonian = Hamiltonian(
        DFTHCHamiltonianContainer(
            one_body_integrals=(one_body + one_body.T) / 2,
            u_matrices=basis_vectors.ravel(),
            w_matrices=0.1 * rng.standard_normal((r, b, c)).ravel(),
            wb_matrix=0.1 * rng.standard_normal((r, c)),
            orbitals=ModelOrbitals(n),
            core_energy=0.0,
            inactive_fock_matrix=np.zeros((n, n)),
        )
    )
    fe2s2_operator = create("qubit_mapper", "sum_of_squares").run(
        synthetic_fe2_s2_hamiltonian, MajoranaMapping.jordan_wigner(2 * n)
    )

    container = synthetic_fe2_s2_hamiltonian.get_container()
    configuration = Configuration.canonical_hf_configuration(
        (FE2S2_PARAMETERS["electrons"] + 1) // 2,
        FE2S2_PARAMETERS["electrons"] // 2,
        container.get_num_orbitals(),
    )
    reference = Wavefunction(
        StateVectorContainer(configuration, container.get_orbitals())
    )
    fe2s2_state_preparation = create("state_prep", "sparse_isometry").run(reference)

    sossa_unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "sossa")
    fe2s2_circuit = create(
        "qpe_circuit_builder",
        "qdk_unary",
        num_queries=fe2s2_num_queries,
        circuit_mapper=AlgorithmRef(
            "circuit_mapper",
            "sossa",
            outer_prepare_algorithm=AlgorithmRef("state_prep", "alias_sampling"),
            inner_prepare_algorithm="controlled_alias_sampling",
            select_algorithm="qrom_phase_gradient",
            rotation_bit_precision=FE2S2_PARAMETERS["b_rot"],
            coefficient_bit_precision=FE2S2_PARAMETERS["b_coeff"],
        ),
        unitary_builder=sossa_unitary_builder,
    ).run(
        state_preparation=fe2s2_state_preparation,
        qubit_hamiltonian=fe2s2_operator,
    )[
        0
    ]
    return fe2s2_circuit.get_qre_application()

In [2]:
from time import perf_counter
from pathlib import Path
from IPython.display import display, HTML

from qdk import Context, qsharp
from qdk.qre import DemoModel1, SequentialCountingProfiler, TemporalProfiler, TemporalProfilerConfig, estimate
from qdk.qre.models import GateBased, RoundBasedFactory, SurfaceCode, TwoDimensionalYokedSurfaceCode
from qdk.qre.application import QSharpApplication, QIRApplication
from qdk.qre.profiler import QREProfiler


REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "library/arithmetic/qsharp.json").is_file()
)
PROFILE_DIR = REPO_ROOT / "source/qep/profile_viewer/profiles"

def make_profiler(record_trace=False, aggregation_window_sec=1.0,t_factories_count=10):
    model = DemoModel1(
        clifford_gate_time_nanos=5000,
        measurment_time_nanos=1000,    
        t_factories_count=t_factories_count,
        ecc_distance=9,
    )
    config = TemporalProfilerConfig(
        seed=42,
        record_trace=record_trace,
        aggregation_window_nanos = int(aggregation_window_sec*1e9),
    )
    profiler = TemporalProfiler(model, config)
    profiler.add_adapter("transpiler")
    profiler.add_adapter("result_adapter")
    return QREProfiler(profiler)

def report_result(result, name, t0=None):
    if t0 is not None:
        dt = perf_counter() - t0
        count = result.totals.total_operations_count
        ns_per_op = int(1e9 * dt / count)    
        print(f"Profiling time {dt:.2f}s, {ns_per_op} ns/op")

    print(f"Estimated runtime: {result.totals.total_runtime_sec}s")
    print("Logical qubits:", result.totals.logical_qubits)
    print("  of them, logical compute qubits:", result.totals.logical_compute_qubits)
    print("Physical qubits:", result.totals.physical_qubits)
    print("Error:", result.totals.error)
    
    json_path = PROFILE_DIR / (name + ".json")
    result.to_json(json_path)
    print(f"Profile saved to {json_path}")
    display(HTML(f'<a href="http://localhost:3000/source/qep/profile_viewer/?profile_name={name}">View profile</a>'))

In [4]:
app = get_sossa_app(num_queries_override=20)
profiler = make_profiler()
t0 = perf_counter()
result = profiler.profile(app, use_qir=False)
report_result(result, f"sossa1", t0=t0)

Changed num_queries from 10162 to 20
[2026-11-10 17:45:17.024057] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] After filtering, only 1 determinant remains, using single reference state preparation
Profiling time 27.05s, 4524 ns/op
Estimated runtime: 17.49672s
Logical qubits: 518
  of them, logical compute qubits: 468
Physical qubits: 83398
Error: 0.0021071911412332543
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/sossa1.json
